# RAPIDS GPU vs CPU Benchmark

Based on [NVIDIA's official cuDF performance comparison](https://github.com/rapidsai/cudf/blob/release/26.04/docs/cudf/source/user_guide/performance-comparisons/performance-comparisons.ipynb)

Demonstrates massive GPU speedups on DataFrame operations, string processing, and ML workloads.

In [ ]:
import time
import os
import timeit
import numpy as np
import cupy as cp
import cudf
import pandas as pd

print("=" * 60)
print("HARDWARE")
print("=" * 60)
cpu_model = "Unknown"
with open("/proc/cpuinfo") as f:
    for line in f:
        if line.startswith("model name"):
            cpu_model = line.split(":")[1].strip()
            break
print(f"CPU: {cpu_model} ({os.cpu_count()} cores)")

import torch
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

print(f"\ncuDF: {cudf.__version__}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

## 1. DataFrame Operations: cuDF vs pandas

Operations on 100M rows. cuDF keeps data in GPU memory and processes all rows in parallel.

Reference: [NVIDIA cuDF performance comparison](https://docs.rapids.ai/api/cudf/stable/user_guide/performance-comparisons/performance-comparisons)

In [ ]:
num_rows = 100_000_000
rng = np.random.default_rng(seed=42)

print(f"Creating DataFrame with {num_rows:,} rows...")
pdf = pd.DataFrame({
    "numbers": rng.integers(-1000, 1000, num_rows, dtype="int64"),
    "category": rng.choice(["Apple", "Google", "Microsoft", "NVIDIA"], size=num_rows),
})
gdf = cudf.from_pandas(pdf)
print(f"Done. DataFrame size: {pdf.memory_usage(deep=True).sum() / 1e9:.2f} GB\n")

results = {}

# --- value_counts ---
t = timeit.timeit(lambda: pdf.value_counts(), number=5)
cpu_time = t / 5
t = timeit.timeit(lambda: gdf.value_counts(), number=5)
gpu_time = t / 5
results['value_counts'] = cpu_time / gpu_time
print(f"value_counts:  CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {results['value_counts']:.0f}x speedup")

# --- groupby agg ---
t = timeit.timeit(lambda: pdf.groupby("category").agg(["min", "max", "mean"]), number=5)
cpu_time = t / 5
t = timeit.timeit(lambda: gdf.groupby("category").agg(["min", "max", "mean"]), number=5)
gpu_time = t / 5
results['groupby'] = cpu_time / gpu_time
print(f"groupby agg:   CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {results['groupby']:.0f}x speedup")

# --- sorting ---
t = timeit.timeit(lambda: pdf.sort_values("numbers"), number=3)
cpu_time = t / 3
t = timeit.timeit(lambda: gdf.sort_values("numbers"), number=3)
gpu_time = t / 3
results['sort'] = cpu_time / gpu_time
print(f"sort:          CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {results['sort']:.0f}x speedup")

print(f"\n{'Operation':<15} {'Speedup':>10}")
print("-" * 27)
for op, speedup in results.items():
    print(f"{op:<15} {speedup:>8.0f}x")

## 2. String Operations (The Biggest Wins)

String processing is where GPU shows the most extreme speedups. Each character operation
on each row is independent — with 100M strings, the GPU has billions of parallel tasks.

NVIDIA's official benchmarks show **up to 1,974x** speedup on string operations.

In [ ]:
import gc
del pdf, gdf
gc.collect()

num_rows = 50_000_000
print(f"Creating string Series with {num_rows:,} rows...")

pd_series = pd.Series(
    rng.choice(["hello world", "123.456", "NVIDIA Tesla T4", "rapids-ai", "gpu_computing"], size=num_rows)
)
gd_series = cudf.from_pandas(pd_series)
print(f"Done.\n")

str_results = {}

# --- str.upper ---
t = timeit.timeit(lambda: pd_series.str.upper(), number=3)
cpu_time = t / 3
t = timeit.timeit(lambda: gd_series.str.upper(), number=3)
gpu_time = t / 3
str_results['str.upper'] = cpu_time / gpu_time
print(f"str.upper:     CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {str_results['str.upper']:.0f}x speedup")

# --- str.contains (regex) ---
t = timeit.timeit(lambda: pd_series.str.contains(r"[0-9][a-z]"), number=3)
cpu_time = t / 3
t = timeit.timeit(lambda: gd_series.str.contains(r"[0-9][a-z]"), number=3)
gpu_time = t / 3
str_results['str.contains'] = cpu_time / gpu_time
print(f"str.contains:  CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {str_results['str.contains']:.0f}x speedup")

# --- str.len ---
t = timeit.timeit(lambda: pd_series.str.len(), number=3)
cpu_time = t / 3
t = timeit.timeit(lambda: gd_series.str.len(), number=3)
gpu_time = t / 3
str_results['str.len'] = cpu_time / gpu_time
print(f"str.len:       CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {str_results['str.len']:.0f}x speedup")

# --- str.replace ---
t = timeit.timeit(lambda: pd_series.str.replace("a", "X"), number=3)
cpu_time = t / 3
t = timeit.timeit(lambda: gd_series.str.replace("a", "X"), number=3)
gpu_time = t / 3
str_results['str.replace'] = cpu_time / gpu_time
print(f"str.replace:   CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {str_results['str.replace']:.0f}x speedup")

# --- str.isalpha (highest speedup — returns bool, no string allocation) ---
t = timeit.timeit(lambda: pd_series.str.isalpha(), number=3)
cpu_time = t / 3
t = timeit.timeit(lambda: gd_series.str.isalpha(), number=3)
gpu_time = t / 3
str_results['str.isalpha'] = cpu_time / gpu_time
print(f"str.isalpha:   CPU={cpu_time:.3f}s  GPU={gpu_time:.4f}s  → {str_results['str.isalpha']:.0f}x speedup")

print(f"\n{'Operation':<15} {'Speedup':>10}")
print("-" * 27)
for op, speedup in str_results.items():
    print(f"{op:<15} {speedup:>8.0f}x")

## 3. Machine Learning: KMeans Clustering

KMeans iteratively computes distances from all points to all cluster centers.
With 2M points and 500 clusters, each iteration involves 1 billion distance calculations.

In [ ]:
del pd_series, gd_series
gc.collect()

from cuml.cluster import KMeans as cuKMeans
from sklearn.cluster import KMeans as skKMeans

n_samples = 2_000_000
n_features = 128
n_clusters = 500
max_iter = 50

print(f"KMeans: {n_samples:,} samples x {n_features} features, {n_clusters} clusters, {max_iter} iterations")
print(f"Each iteration: {n_samples:,} x {n_clusters} = {n_samples*n_clusters:,} distance computations\n")

np.random.seed(42)
X_cpu = np.random.rand(n_samples, n_features).astype(np.float32)
X_gpu = cp.asarray(X_cpu)

# --- GPU ---
start = time.perf_counter()
cu_km = cuKMeans(n_clusters=n_clusters, max_iter=max_iter, random_state=42)
cu_km.fit(X_gpu)
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
print(f"GPU (cuML): {gpu_time:.2f}s")

# --- CPU ---
start = time.perf_counter()
sk_km = skKMeans(n_clusters=n_clusters, max_iter=max_iter, random_state=42, n_init=1)
sk_km.fit(X_cpu)
cpu_time = time.perf_counter() - start
print(f"CPU (sklearn): {cpu_time:.2f}s")

print(f"\n>>> Speedup: {cpu_time/gpu_time:.1f}x faster on GPU <<<")

## 4. Random Forest Training

100 decision trees, each built independently — perfect for GPU parallelism.
The GPU builds all trees and evaluates all node splits concurrently.

In [ ]:
del X_cpu, X_gpu
gc.collect()

from cuml.ensemble import RandomForestClassifier as cuRF
from sklearn.ensemble import RandomForestClassifier as skRF

n_samples = 1_000_000
n_features = 64
n_classes = 10
n_trees = 100

print(f"Random Forest: {n_samples:,} samples, {n_features} features, {n_classes} classes, {n_trees} trees\n")

np.random.seed(42)
X_cpu = np.random.rand(n_samples, n_features).astype(np.float32)
y_cpu = np.random.randint(0, n_classes, n_samples).astype(np.int32)
X_gpu = cp.asarray(X_cpu)
y_gpu = cp.asarray(y_cpu)

# --- GPU ---
start = time.perf_counter()
gpu_rf = cuRF(n_estimators=n_trees, max_depth=12, random_state=42)
gpu_rf.fit(X_gpu, y_gpu)
cp.cuda.Stream.null.synchronize()
gpu_time = time.perf_counter() - start
print(f"GPU (cuML): {gpu_time:.2f}s")

# --- CPU ---
start = time.perf_counter()
cpu_rf = skRF(n_estimators=n_trees, max_depth=12, random_state=42, n_jobs=-1)
cpu_rf.fit(X_cpu, y_cpu)
cpu_time = time.perf_counter() - start
print(f"CPU (sklearn, all cores): {cpu_time:.2f}s")

print(f"\n>>> Speedup: {cpu_time/gpu_time:.1f}x faster on GPU <<<")

## Summary

| Category | Operation | Expected Speedup |
|----------|-----------|------------------|
| **DataFrame** | value_counts, groupby, sort | 30-170x |
| **Strings** | upper, contains, replace, len | 100-2000x |
| **ML** | KMeans clustering | 20-50x |
| **ML** | Random Forest training | 10-30x |

### Why strings show the biggest speedup

String operations on pandas are Python-level loops over each row (no vectorization).
cuDF processes all 50M strings in parallel using GPU kernels — each string on its own thread.
This is the ideal GPU workload: millions of tiny independent tasks.

### References

- [NVIDIA cuDF Performance Comparisons](https://docs.rapids.ai/api/cudf/stable/user_guide/performance-comparisons/performance-comparisons)
- [NVIDIA/workbench-example-rapids-cudf](https://github.com/NVIDIA/workbench-example-rapids-cudf)